# Volcano ELT Pipeline

Ten pipeline składa się z trzech kroków:
- **Extract** — pobieram dane z zewnętrznych źródeł (API, pliki)
- **Load** — zapisuję dane surowe na dysk (nasz lokalny "data lake")
- **Transform** — czyszczę i przekształcay dane w pandas

Źródła:
1. **NASA EONET API** — bieżące zdarzenia wulkaniczne (ostatnie 2 lata)
2. **NOAA NGDC API** — historyczne znaczące erupcje wulkanów

In [1]:
import requests

# EXTRACT #1 — NASA EONET: bieżące zdarzenia wulkaniczne

NASA EONET (Earth Observatory Natural Event Tracker) śledzi naturalne zdarzenia na Ziemi w czasie rzeczywistym.

Parametry zapytania:
- `category=volcanoes` — tylko zdarzenia wulkaniczne
- `status=all` — zarówno aktywne jak i zakończone
- `days=730` — ostatnie 2 lata

In [2]:
import json

EONET_URL = "https://eonet.gsfc.nasa.gov/api/v3/events"

params = {
    "category": "volcanoes",
    "status": "all",
    "days": 730
}

print("Pobieranie danych z NASA EONET...")
response = requests.get(EONET_URL, params=params)
# raise_for_status() rzuca błąd jeśli request się nie powiódł (np. 404, 500)
response.raise_for_status()

eonet_raw = response.json()

print(f"Status: {response.status_code}")
print(f"Pobrano zdarzeń: {len(eonet_raw['events'])}")
print(f"\nPrzykładowe zdarzenie:")
print(json.dumps(eonet_raw['events'][0], indent = 4))

Pobieranie danych z NASA EONET...
Status: 200
Pobrano zdarzeń: 71

Przykładowe zdarzenie:
{
    "id": "EONET_20710",
    "title": "Nevados del Chillan Volcano, Chile",
    "description": null,
    "link": "https://eonet.gsfc.nasa.gov/api/v3/events/EONET_20710",
    "closed": null,
    "categories": [
        {
            "id": "volcanoes",
            "title": "Volcanoes"
        }
    ],
    "sources": [
        {
            "id": "SIVolcano",
            "url": "https://volcano.si.edu/volcano.cfm?vn=357070"
        }
    ],
    "geometry": [
        {
            "magnitudeValue": null,
            "magnitudeUnit": null,
            "date": "2026-06-15T00:00:00Z",
            "type": "Point",
            "coordinates": [
                -71.378,
                -36.868
            ]
        }
    ]
}


In [3]:
len(response.json()['events'])

71

In [4]:
import sys
sys.path.append('..')
import config

print(f'NOAA NGDC URL: {config.NOAA_NGDC}')
response = requests.get(config.NOAA_NGDC)
response.json()



NOAA NGDC URL: https://www.ngdc.noaa.gov/hazel/hazard-service/api/v1/volcanoes


{'items': [{'id': 1,
   'year': 1169,
   'month': 2,
   'day': 4,
   'tsunamiEventId': 2852,
   'earthquakeEventId': 421,
   'volcanoLocationId': 10106,
   'volcanoLocationNewNum': 211060,
   'volcanoLocationNum': '0101-06=',
   'name': 'Etna',
   'location': 'Italy',
   'country': 'Italy',
   'latitude': 37.748,
   'longitude': 14.999,
   'elevation': 3357,
   'morphology': 'Stratovolcano',
   'agent': 'S,W',
   'deathsTotal': 16000,
   'deathsAmountOrderTotal': 4,
   'damageAmountOrderTotal': 4,
   'significant': True,
   'publish': False,
   'eruption': True,
   'status': 'Historical',
   'timeErupt': 'D1'},
  {'id': 2,
   'year': 1329,
   'month': 7,
   'day': 15,
   'volcanoLocationId': 10106,
   'volcanoLocationNewNum': 211060,
   'volcanoLocationNum': '0101-06=',
   'name': 'Etna',
   'location': 'Italy',
   'country': 'Italy',
   'latitude': 37.748,
   'longitude': 14.999,
   'elevation': 3357,
   'morphology': 'Stratovolcano',
   'vei': 3,
   'agent': 'I',
   'deathsAmountOrde

In [5]:
len(response.json()['items'])

200

In [6]:
data = response.json()
print(data['itemsPerPage'])

200


In [7]:
print({k: v for k, v in data.items() if k != 'items'})

{'page': 1, 'totalPages': 5, 'itemsPerPage': 200, 'totalItems': 900}


In [8]:
volcanoes = []

for pageIndex in range (1, data['totalPages'] + 1):
    response = requests.get(config.NOAA_NGDC, params={'page': pageIndex, 'pageSize': config.NOAA_PAGE_SIZE})
    volcanoes.extend(response.json()['items'])
 
len(volcanoes)


JSONDecodeError: Expecting value: line 1 column 1 (char 0)